# Semantic Search End-to-End (LIVE DEMO)

**Purpose:** A simple, reliable, visually satisfying end-to-end semantic search demo.

We will:
1. Frame the problem: **keyword search fails when words differ**.
2. Load a medium-small dataset (**200–1,000 short snippets**).
3. Build embeddings + a FAISS index.
4. Implement an interactive `search(query, k=5)` function.
5. Run **5 curated “wow” queries** that show semantic retrieval with minimal keyword overlap.
6. Discuss: *“Why did this match?”*

---

## Key idea

> Semantic search retrieves by **meaning**, not by exact word overlap.


## 0) Setup

We’ll use:
- `sentence-transformers` for embeddings
- `faiss` for fast nearest-neighbor search
- `scikit-learn` TF-IDF for a simple keyword baseline (so we can compare)

Install notes:
- `faiss-cpu` works well for CPU demos.


In [1]:
# Install dependencies if needed (safe to re-run)
try:
    import sentence_transformers  # noqa: F401
except ImportError:
    !uv add sentence-transformers

try:
    import faiss  # noqa: F401
except ImportError:
    !uv add faiss-cpu

try:
    import sklearn  # noqa: F401
except ImportError:
    !uv add scikit-learn

In [2]:
import re
import random
import numpy as np
import pandas as pd

import faiss
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 140)
random.seed(42)
np.random.seed(42)

## 1) Problem framing: keyword search fails when words differ

Keyword search relies on **exact words** (or close variants).

Example:
- Query: “**cut back on expenses**”
- Text: “**reduce spending**”

These mean the same thing, but a pure keyword search may miss it if overlap is low.


## 2) Build a medium-small dataset (200–1,000 short snippets)

For a live demo, we want:
- short, readable texts
- multiple topics
- lots of paraphrases / synonyms

We’ll generate ~300 snippets using simple templates across domains.


In [3]:
def make_snippets(n_per_topic=30):
    topics = {}

    topics["finance"] = [
        "Lower your monthly costs by reducing spending on non-essentials.",
        "Track your cash flow to understand where your money goes.",
        "Build an emergency fund by setting aside a small amount weekly.",
        "Pay down high-interest debt first to save on interest payments.",
        "Create a budget to control expenses without feeling deprived.",
        "Negotiate your bills to decrease recurring charges.",
        "Automate savings transfers so you don't forget.",
        "Diversify investments to manage portfolio risk.",
        "Review subscriptions you no longer use and cancel them.",
        "Set financial goals and monitor progress every month.",
    ]

    topics["software"] = [
        "Speed up a slow script by profiling hotspots and optimizing bottlenecks.",
        "Refactor messy code to reduce maintenance pain later.",
        "Add caching to avoid recomputing expensive results.",
        "Use indexes in your database to improve query performance.",
        "Write tests to prevent regressions when you change logic.",
        "Reduce latency by batching requests instead of sending many small calls.",
        "Vectorize operations in NumPy for faster computation.",
        "Use pagination when retrieving large datasets from APIs.",
        "Clean up technical debt to make future changes safer.",
        "Improve reliability by adding retries and timeouts.",
    ]

    topics["ml_ai"] = [
        "Embeddings represent text as vectors so similar meanings are close.",
        "Fine-tuning adapts a pretrained model to a specific task.",
        "A classifier predicts labels from features in your dataset.",
        "Overfitting happens when a model memorizes training data.",
        "Normalize inputs to stabilize training dynamics.",
        "Use cross-validation to estimate generalization performance.",
        "RAG combines retrieval with generation for grounded answers.",
        "Prompting can steer outputs without changing model weights.",
        "A confusion matrix summarizes classification errors.",
        "Dimensionality reduction helps visualize high-dimensional data.",
    ]

    topics["health"] = [
        "Drink more water to stay hydrated throughout the day.",
        "Walking daily can improve cardiovascular health.",
        "Prioritize sleep to support memory and focus.",
        "Lower stress by doing short breathing exercises.",
        "Eat more vegetables for better nutrition.",
        "Strength training supports bone density over time.",
        "Avoid sugary drinks to reduce empty calories.",
        "Stretch after workouts to improve flexibility.",
        "Schedule regular checkups to catch issues early.",
        "Balance meals with protein, fiber, and healthy fats.",
    ]

    topics["travel"] = [
        "Pack light to make airport transfers easier.",
        "Arrive early to avoid missing your flight.",
        "Check weather forecasts before going on a trip.",
        "Use offline maps when traveling with limited data.",
        "Bring a portable charger for long travel days.",
        "Choose travel insurance to cover unexpected cancellations.",
        "Plan routes to avoid heavy traffic during peak hours.",
        "Keep photocopies of important documents while abroad.",
        "Book accommodations near public transport for convenience.",
        "Learn a few local phrases to navigate more smoothly.",
    ]

    topics["weather"] = [
        "A tropical storm can intensify quickly over warm waters.",
        "Strong winds and heavy rain may cause flooding in low areas.",
        "Monitor official advisories when a cyclone is nearby.",
        "Secure loose objects before severe weather arrives.",
        "Storm surges can be more dangerous than wind in coastal zones.",
        "Evacuate early if authorities issue warnings for your area.",
        "Prepare emergency supplies like water, food, and batteries.",
        "Avoid driving through flooded roads during heavy rain.",
        "Thunderstorms can produce lightning and sudden gusts.",
        "Heat waves increase the risk of dehydration and exhaustion.",
    ]

    topics["food"] = [
        "Try adding chili oil to ramen for extra spice.",
        "Marinate chicken to make it more flavorful and tender.",
        "Use fresh herbs to brighten a simple dish.",
        "Simmer soup slowly to deepen the taste.",
        "Balance sweet and sour flavors in sauces.",
        "Roast vegetables to bring out their natural sweetness.",
        "Taste as you cook and adjust seasoning gradually.",
        "Let steak rest before slicing to keep it juicy.",
        "Use high-quality olive oil for salads and dressings.",
        "Bake bread until the crust turns golden brown.",
    ]

    topics["business"] = [
        "Customer retention improves when support resolves issues quickly.",
        "A loyalty program can increase repeat purchases.",
        "Segment users to tailor messaging by needs and behavior.",
        "Reduce churn by identifying early warning signals.",
        "Improve conversion by simplifying onboarding steps.",
        "Measure campaign impact with controlled experiments.",
        "Pricing changes should be tested to avoid unintended losses.",
        "Forecast demand to prevent stockouts during peak seasons.",
        "Analyze feedback to understand why customers leave.",
        "Align teams on shared outcomes rather than isolated metrics.",
    ]

    # Expand via paraphrase-like variations
    variants = [
        ("Lower", "Decrease"),
        ("reduce", "cut"),
        ("improve", "boost"),
        ("avoid", "prevent"),
        ("plan", "map out"),
        ("monitor", "keep an eye on"),
        ("quickly", "rapidly"),
        ("support", "help"),
        ("issues", "problems"),
        ("customers", "clients"),
    ]

    snippets = []
    for topic, base in topics.items():
        for _ in range(n_per_topic):
            s = random.choice(base)
            # Apply 0-2 random word substitutions to create light paraphrases
            s2 = s
            for _k in range(random.randint(0, 2)):
                a, b = random.choice(variants)
                s2 = re.sub(rf"\b{re.escape(a)}\b", b, s2, flags=re.IGNORECASE)
                s2 = re.sub(rf"\b{re.escape(b)}\b", a, s2, flags=re.IGNORECASE)
            snippets.append({"topic": topic, "text": s2})

    random.shuffle(snippets)
    return snippets

data = make_snippets(n_per_topic=38)  # 8 topics * 38 = 304 snippets
df = pd.DataFrame(data)
df.insert(0, "id", [f"D{i:04d}" for i in range(len(df))])

df.shape, df.head()

((304, 3),
       id     topic  \
 0  D0000   finance   
 1  D0001  business   
 2  D0002     ml_ai   
 3  D0003   finance   
 4  D0004    travel   
 
                                                               text  
 0  Pay down high-interest debt first to save on interest payments.  
 1     Pricing changes should be tested to avoid unintended losses.  
 2  Dimensionality reduction helps visualize high-dimensional data.  
 3  Pay down high-interest debt first to save on interest payments.  
 4               Use offline maps when traveling with limited data.  )

## 3) Build embeddings + FAISS index

We’ll use **normalized embeddings** and an **inner-product index**:

- Normalize embeddings → vectors have length 1
- Inner product ≈ cosine similarity
- Higher score = more similar

This makes the ranking easy to interpret for semantic search demos.


In [4]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(model_name)

texts = df["text"].tolist()
emb = model.encode(texts, normalize_embeddings=True).astype("float32")

dim = emb.shape[1]
index = faiss.IndexFlatIP(dim)  # inner product (with normalized vectors ~ cosine)
index.add(emb)

print("Embeddings shape:", emb.shape)
print("FAISS index size:", index.ntotal)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embeddings shape: (304, 384)
FAISS index size: 304


## 4) Keyword baseline (TF-IDF)

This is our **keyword-ish** retrieval baseline:
- TF-IDF vectors + cosine similarity
- Relies heavily on word overlap
- Not “classic keyword search” (like BM25), but good enough to show the failure mode


In [5]:
tfidf = TfidfVectorizer(stop_words="english")
X_tfidf = tfidf.fit_transform(texts)  # sparse matrix
X_tfidf.shape

(304, 387)

## 5) Interactive search function

### `search(query, k=5)` returns:
- semantic results (FAISS)
- keyword baseline results (TF-IDF)

We’ll also show a quick “keyword overlap” hint (shared tokens) so you can point out
when semantic results succeed with minimal overlap.


In [6]:
def _tokenize(s: str):
    return set(re.findall(r"[a-zA-Z]+", s.lower()))

def semantic_search(query, k=5):
    q = model.encode([query], normalize_embeddings=True).astype("float32")
    scores, idx = index.search(q, k)
    rows = []
    q_tokens = _tokenize(query)
    for rank, (i, score) in enumerate(zip(idx[0], scores[0]), start=1):
        text = df.loc[i, "text"]
        overlap = len(q_tokens & _tokenize(text))
        rows.append({
            "rank": rank,
            "id": df.loc[i, "id"],
            "topic": df.loc[i, "topic"],
            "score(cos≈ip)": float(score),
            "overlap_tokens": overlap,
            "text": text
        })
    return pd.DataFrame(rows)

def keyword_search(query, k=5):
    qv = tfidf.transform([query])
    sims = cosine_similarity(qv, X_tfidf).ravel()
    top = np.argsort(-sims)[:k]
    rows = []
    q_tokens = _tokenize(query)
    for rank, i in enumerate(top, start=1):
        text = df.loc[i, "text"]
        overlap = len(q_tokens & _tokenize(text))
        rows.append({
            "rank": rank,
            "id": df.loc[i, "id"],
            "topic": df.loc[i, "topic"],
            "score(tfidf_cos)": float(sims[i]),
            "overlap_tokens": overlap,
            "text": text
        })
    return pd.DataFrame(rows)

def search(query, k=5):
    print("QUERY:", query)
    print("\n--- Semantic (FAISS) ---")
    display(semantic_search(query, k=k))
    print("\n--- Keyword baseline (TF-IDF) ---")
    display(keyword_search(query, k=k))

# quick sanity run
search("How can I save money by spending less?", k=5)

QUERY: How can I save money by spending less?

--- Semantic (FAISS) ---


,rank,id,topic,score(cos≈ip),overlap_tokens,text
0,1,D0273,finance,0.692708,2,Lower your monthly costs by reducing spending on non-essentials.
1,2,D0075,finance,0.692708,2,Lower your monthly costs by reducing spending on non-essentials.
2,3,D0125,finance,0.641155,0,Create a budget to control expenses without feeling deprived.
3,4,D0117,finance,0.641155,0,Create a budget to control expenses without feeling deprived.
4,5,D0049,finance,0.641155,0,Create a budget to control expenses without feeling deprived.



--- Keyword baseline (TF-IDF) ---


,rank,id,topic,score(tfidf_cos),overlap_tokens,text
0,1,D0000,finance,0.253245,1,Pay down high-interest debt first to save on interest payments.
1,2,D0300,finance,0.253245,1,Pay down high-interest debt first to save on interest payments.
2,3,D0236,finance,0.253245,1,Pay down high-interest debt first to save on interest payments.
3,4,D0089,finance,0.253245,1,Pay down high-interest debt first to save on interest payments.
4,5,D0003,finance,0.253245,1,Pay down high-interest debt first to save on interest payments.


## 6) Showcase: 5 curated “wow” queries

These are chosen to demonstrate:
- Meaning-based matches
- Minimal keyword overlap
- Cross-topic disambiguation

Run the next cell live and ask:
> “Why did this match?”


In [7]:
wow_queries = [
    # Finance: "cut back" vs "reduce spending" etc.
    "How do I cut back on expenses without feeling miserable?",
    
    # Software performance: "slow" vs "optimize bottlenecks/profiling"
    "My program is slow. How do I make it run faster?",
    
    # Weather preparedness: "storm" vs "cyclone/typhoon advisories"
    "A big storm is coming. What should I do to prepare?",
    
    # Business retention: "keep customers" vs "reduce churn / loyalty"
    "How can a company keep customers from leaving?",
    
    # Health: "stress" vs "breathing exercises / sleep"
    "I feel anxious and stressed—what helps right away?",
]

for q in wow_queries:
    print("="*90)
    search(q, k=5)

QUERY: How do I cut back on expenses without feeling miserable?

--- Semantic (FAISS) ---


,rank,id,topic,score(cos≈ip),overlap_tokens,text
0,1,D0125,finance,0.627445,3,Create a budget to control expenses without feeling deprived.
1,2,D0117,finance,0.627445,3,Create a budget to control expenses without feeling deprived.
2,3,D0170,finance,0.627445,3,Create a budget to control expenses without feeling deprived.
3,4,D0049,finance,0.627445,3,Create a budget to control expenses without feeling deprived.
4,5,D0018,finance,0.627445,3,Create a budget to control expenses without feeling deprived.



--- Keyword baseline (TF-IDF) ---


,rank,id,topic,score(tfidf_cos),overlap_tokens,text
0,1,D0049,finance,0.57735,3,Create a budget to control expenses without feeling deprived.
1,2,D0170,finance,0.57735,3,Create a budget to control expenses without feeling deprived.
2,3,D0117,finance,0.57735,3,Create a budget to control expenses without feeling deprived.
3,4,D0125,finance,0.57735,3,Create a budget to control expenses without feeling deprived.
4,5,D0018,finance,0.57735,3,Create a budget to control expenses without feeling deprived.


QUERY: My program is slow. How do I make it run faster?

--- Semantic (FAISS) ---


...

,rank,id,topic,score(tfidf_cos),overlap_tokens,text
0,1,D0218,business,0.460734,1,Analyze feedback to understand why customers leave.
1,2,D0156,business,0.460734,1,Analyze feedback to understand why customers leave.
2,3,D0159,business,0.460734,1,Analyze feedback to understand why customers leave.
3,4,D0256,business,0.460734,1,Analyze feedback to understand why customers leave.
4,5,D0277,business,0.460734,1,Analyze feedback to understand why customers leave.


QUERY: I feel anxious and stressed—what helps right away?

--- Semantic (FAISS) ---


,rank,id,topic,score(cos≈ip),overlap_tokens,text
0,1,D0243,health,0.535913,0,Lower stress by doing short breathing exercises.
1,2,D0237,health,0.535913,0,Lower stress by doing short breathing exercises.
2,3,D0097,health,0.535913,0,Lower stress by doing short breathing exercises.
3,4,D0066,health,0.535913,0,Lower stress by doing short breathing exercises.
4,5,D0046,health,0.535913,0,Lower stress by doing short breathing exercises.



--- Keyword baseline (TF-IDF) ---


,rank,id,topic,score(tfidf_cos),overlap_tokens,text
0,1,D0053,ml_ai,0.401828,1,Dimensionality reduction helps visualize high-dimensional data.
1,2,D0123,ml_ai,0.401828,1,Dimensionality reduction helps visualize high-dimensional data.
2,3,D0002,ml_ai,0.401828,1,Dimensionality reduction helps visualize high-dimensional data.
3,4,D0009,ml_ai,0.401828,1,Dimensionality reduction helps visualize high-dimensional data.
4,5,D0280,weather,0.000000,1,"Prepare emergency supplies like water, food, and batteries."


## 7) Discussion prompts (use these live)

1. **Keyword failure:** For which query did TF-IDF do the worst? Why?
2. **Minimal overlap:** Find a semantic match with **0–1 overlap tokens**. Why did it still match?
3. **Model bias:** What kinds of matches feel “off”? What might the encoder be confusing?
4. **Encoder dependence:** If we change the embedding model, which results do you think will change most?
5. **Similarity ≠ correctness:** Even if a match is similar, is it actually the “right” answer for the user?


## Outputs checklist

- ✅ Clean semantic matches with minimal keyword overlap
- ✅ An interactive `search(query, k=5)` function
- ✅ 5 curated “wow” queries for live demo
